In [1]:
import pandas as pd

In [2]:
data = {
    'user_id': [1, 1, 1, 2, 2, 2],
    'timestamp': ['10:00', '10:01', '10:02', '10:00', '10:01', '10:02'],
    'ping_ms': [20, 22, 21, 30, 32, 150] # User 2 spikes at 10:02
}
df = pd.DataFrame(data)
df

,user_id,timestamp,ping_ms
0,1,10:00,20
1,1,10:01,22
2,1,10:02,21
3,2,10:00,30
4,2,10:01,32
5,2,10:02,150


In [3]:
type(df)

pandas.DataFrame

In [4]:
df = df.sort_values(by=['user_id', 'timestamp'])

In [5]:
df

,user_id,timestamp,ping_ms
0,1,10:00,20
1,1,10:01,22
2,1,10:02,21
3,2,10:00,30
4,2,10:01,32
5,2,10:02,150


In [6]:
df['ping_spike'] = df.groupby('user_id')['ping_ms'].diff()

In [7]:
# df.drop(columns=['ping_difference'], axis = 0, inplace=True)
# df

In [8]:
df

,user_id,timestamp,ping_ms,ping_spike
0,1,10:00,20,NaN
1,1,10:01,22,2.0
2,1,10:02,21,-1.0
3,2,10:00,30,NaN
4,2,10:01,32,2.0
5,2,10:02,150,118.0


In [9]:
anomalies = df[df['ping_spike'] > 50]
anomalies

,user_id,timestamp,ping_ms,ping_spike
5,2,10:02,150,118.0


In [10]:
# df['regional_rank'] = df.groupby('region')['session_duration_minutes'].rank(method='dense', ascending=False)

In [11]:
import pandas as pd

data = {
    'session_id': [101, 102, 103, 104, 105, 106, 107],
    'region': ['US-West', 'US-West', 'US-West', 'EU-Central', 'EU-Central', 'US-East', 'US-East'],
    'user_id': [1, 2, 3, 4, 5, 6, 7],
    'duration_minutes': [120, 120, 45, 90, 85, 200, 15] #tie at 120 in US-West
}
df = pd.DataFrame(data)

# 2. Apply DENSE_RANK partitioned by region (groupby) and ordered descending
df['regional_rank'] = df.groupby('region')['duration_minutes'].rank(method='dense', ascending=False)

# 3. Filter for only the top 2 longest sessions per region
top_sessions = df[df['regional_rank'] <= 2].sort_values(by=['region', 'regional_rank'])

print("Full DataFrame with Ranks:")
print(df.sort_values(by=['region', 'regional_rank']))
print("\n---")
print("Top 2 Sessions Per Region:")
print(top_sessions)

Full DataFrame with Ranks:
   session_id      region  user_id  duration_minutes  regional_rank
3         104  EU-Central        4                90            1.0
4         105  EU-Central        5                85            2.0
5         106     US-East        6               200            1.0
6         107     US-East        7                15            2.0
0         101     US-West        1               120            1.0
1         102     US-West        2               120            1.0
2         103     US-West        3                45            2.0

---
Top 2 Sessions Per Region:
   session_id      region  user_id  duration_minutes  regional_rank
3         104  EU-Central        4                90            1.0
4         105  EU-Central        5                85            2.0
5         106     US-East        6               200            1.0
6         107     US-East        7                15            2.0
0         101     US-West        1               120     